# Helper functions (and examples) to align (and view) molecules

In [1]:
from pymatgen.core import Molecule
from pymatgen.analysis import molecule_matcher

def align(base_atoms, target_atoms, threshold=0.5, verbose=False):
    if verbose: print('Convert ASE atoms to Pymatgen molecule')
    base = Molecule.from_ase_atoms(base_atoms)
    target = Molecule.from_ase_atoms(target_atoms)
    
    # Get initial guess
    if verbose: print('Computing initial guess via cheap and naive alignment')
    matcher_guess = molecule_matcher.HungarianOrderMatcher(target)
    aligned_guess, rmsd_guess = matcher_guess.fit(base)
    if verbose: print(f'Hungarian-based alignment yielded: iRMSD={rmsd_guess}')
    
    # Exact matcher
    effective_threshold = min(rmsd_guess+1e-3, threshold)
    if verbose: print(f'Attempting exact match with iRMSD threshold: {effective_threshold}')
    matcher = molecule_matcher.GeneticOrderMatcher(target, threshold=effective_threshold)
    results = matcher.fit(aligned_guess)
    
    # Return best result
    if not results:
        if verbose: print(f'Could not find alignment below iRMSD threshold')
        return(aligned_guess.to_ase_atoms(), None)
    aligned, rmsd = min(results, key=lambda x:x[-1])
    if verbose: print(f'Best alignment found with iRMSD={rmsd}')
    return(aligned.to_ase_atoms(), rmsd)

In [2]:
import ase.io

def write_all_aligned(input_path, output_path, index_target=0, threshold=0.5, verbose=False):
    if verbose: print(f'Reading geometries from {input_path}')
    all_atoms = ase.io.read(input_path, index=':')

    if verbose: print(f'Aligning all {len(all_atoms)} geometries found, w.r.t. to geometry {index_target}, using threshold {threshold}')
    target = all_atoms[index_target]
    all_comments = []
    for current_index, atoms in enumerate(all_atoms):
        # Extract comments
        comments = ' '.join(atoms.info)
        
        # Skip if reference geometry
        if current_index == index_target:
            all_comments.append(comments)
            continue

        # Compute and save aligned geometry
        aligned, rmsd = align(atoms, target, threshold=threshold, verbose=verbose)
        all_atoms[current_index] = aligned
        all_comments.append(f'{comments} (iRMSD={rmsd})')

    if verbose: print(f'Writing all {len(all_atoms)} aligned geometries to {output_path}')
    for current_index, (atoms, comment) in enumerate(zip(all_atoms, all_comments)):
        ase.io.write(output_path, atoms, comment=comment, append=(current_index > 0))

In [3]:
# conda install -c conda-forge ipywidgets nglview
from ase.visualize import view

def view_aligned(aligned, target):
    v = view(aligned+target, viewer='ngl')
    v.view.remove_spacefill()
    # Try having the aligned (generated) geometry with sub 1.0 opacity around the reference geometry.  
    v.view.add_ball_and_stick(selection=range(len(aligned)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.2, radiusType='covalent',opacity=0.7)
    v.view.add_ball_and_stick(selection=range(len(aligned),len(aligned)+len(target)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.1, radiusType='covalent')
    return(v)

# Example on single pair for molecules to align and view

In [4]:
import ase.io

all_atoms = ase.io.read('scratch/results/results/sample_selected_TS-20250709-SCAN-9w-various_models/rcmconly_passerini-TS13605.xyz', index=':')
target = all_atoms[1]
base = all_atoms[-1]
' '.join(base.info)

'Generated/inpainted transition state. RMSD: 1.0 Å. By model /misc/home/guest50/OAReactDiff/oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/9w-cutoff_12-lr5e-4-StepLR-SCAN-leftnet3b7e89f781c1/ddpm-epoch'

In [5]:
aligned_base, rmsd = align(base, target, threshold=0.5, verbose=True)
print(f'iRSMD={rmsd}')

Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=1.0106337301355603
Attempting exact match with iRMSD threshold: 0.5
Could not find alignment below iRMSD threshold
iRSMD=None


In [6]:
view_aligned(aligned_base, target)

# Example to produce aligned XYZs

In [7]:
import os

input_dir = os.path.abspath('scratch/results/results/sample_selected_TS-20250709-SCAN-9w-various_models/')
input_file = 'rcmconly_passerini-TS13605.xyz'
input_path = os.path.join(input_dir, input_file)
output_dir = os.path.join(os.path.dirname(input_dir), os.path.basename(input_dir)+'_aligned_20250724')
os.makedirs(output_dir, exist_ok=True)
output_file = input_file.removesuffix('.xyz') + '_aligned.xyz'
output_path = os.path.join(output_dir, output_file)
print(output_dir)

write_all_aligned(input_path, output_path, index_target=1, threshold=0.5, verbose=True)

/Users/trondlinjordet/Documents/MANABIYA_work/OAReactDiff/scratch/results/results/sample_selected_TS-20250709-SCAN-9w-various_models_aligned_20250724
Reading geometries from /Users/trondlinjordet/Documents/MANABIYA_work/OAReactDiff/scratch/results/results/sample_selected_TS-20250709-SCAN-9w-various_models/rcmconly_passerini-TS13605.xyz
Aligning all 22 geometries found, w.r.t. to geometry 1, using threshold 0.5
Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=1.0636730644672765
Attempting exact match with iRMSD threshold: 0.5
Best alignment found with iRMSD=0.16056616433537615
Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=1.0329416026913314
Attempting exact match with iRMSD threshold: 0.5
Best alignment found with iRMSD=0.3379527773559884
Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and na

# Trond: align the generated multi_block XYZs 
Ground truth expected as second block, i.e., target_index=1. This is because the blocks are expected to be ordered as:
```
Reference reactant
Reference transition state
Reference product
Generated transition state 1
Generated transition state 2
...
```

In [8]:
import os

def align_many_files(input_dir: str, output_dir: str=None, index_target=1, threshold=0.5, verbose=True):
    if output_dir is not None:
        output_dir = input_dir
    files = os.listdir(input_dir)
    files = [x for x in files if os.path.isfile(os.path.join(input_dir, x))]
    files = [x for x in files if x.endswith('.xyz')]
    for input_filename in files:
        output_filename = input_filename.removesuffix('.xyz') + '_aligned.xyz'
        input_path = os.path.join(input_dir, input_filename)
        output_path = os.path.join(output_dir, output_filename)
        write_all_aligned(input_path, output_path, index_target=index_target, threshold=threshold, verbose=verbose)

In [9]:
results_9w_dir = input_dir #'/misc/home/guest50/OAReactDiff/results/sample_selected_TS-20250709-SCAN-9w-various_models'
results_10w_dir = ''

In [10]:
output_dir

'/Users/trondlinjordet/Documents/MANABIYA_work/OAReactDiff/scratch/results/results/sample_selected_TS-20250709-SCAN-9w-various_models_aligned_20250724'

In [11]:
files = os.listdir(results_9w_dir)

In [12]:
files

['WL1-TS6029_aligned_aligned_aligned.xyz',
 'WL1-TS11193.xyz',
 'WL1-TS784_aligned_aligned.xyz',
 'WL1-TS11193_aligned_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS5435_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS5435_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS17328.xyz',
 'rcmconly_passerini-TS7394_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS17328_aligned.xyz',
 'rcmconly_passerini-TS13605.xyz',
 'WL1-TS6029_aligned_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS17605_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS1206_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS7394_aligned.xyz',
 'rcmconly_strecker-TS3978_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS13605_aligned_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS1206_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS7394_aligned_aligned.xyz',
 'rcmconly_passerini-TS1206_aligned_aligned.xyz',
 'WL1-TS6029_aligned.xyz

In [13]:
[x for x in files if os.path.isfile(os.path.join(results_9w_dir, x))]

['WL1-TS6029_aligned_aligned_aligned.xyz',
 'WL1-TS11193.xyz',
 'WL1-TS784_aligned_aligned.xyz',
 'WL1-TS11193_aligned_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS5435_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS5435_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS17328.xyz',
 'rcmconly_passerini-TS7394_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS17328_aligned.xyz',
 'rcmconly_passerini-TS13605.xyz',
 'WL1-TS6029_aligned_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS17605_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS1206_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS7394_aligned.xyz',
 'rcmconly_strecker-TS3978_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS13605_aligned_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS1206_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS7394_aligned_aligned.xyz',
 'rcmconly_passerini-TS1206_aligned_aligned.xyz',
 'WL1-TS6029_aligned.xyz

In [14]:
[x for x in files if x.endswith('.xyz')]

['WL1-TS6029_aligned_aligned_aligned.xyz',
 'WL1-TS11193.xyz',
 'WL1-TS784_aligned_aligned.xyz',
 'WL1-TS11193_aligned_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS5435_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS5435_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS17328.xyz',
 'rcmconly_passerini-TS7394_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS17328_aligned.xyz',
 'rcmconly_passerini-TS13605.xyz',
 'WL1-TS6029_aligned_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS17605_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS1206_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS7394_aligned.xyz',
 'rcmconly_strecker-TS3978_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS13605_aligned_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS1206_aligned_aligned_aligned_aligned.xyz',
 'rcmconly_passerini-TS7394_aligned_aligned.xyz',
 'rcmconly_passerini-TS1206_aligned_aligned.xyz',
 'WL1-TS6029_aligned.xyz

In [15]:
#os.makedirs('/misc/home/guest50/OAReactDiff/results/sample_selected_TS-20250709-SCAN-9w-various_models_aligned', exist_ok=True)

In [16]:
align_many_files(input_dir, output_dir)

Reading geometries from /Users/trondlinjordet/Documents/MANABIYA_work/OAReactDiff/scratch/results/results/sample_selected_TS-20250709-SCAN-9w-various_models/WL1-TS6029_aligned_aligned_aligned.xyz
Aligning all 22 geometries found, w.r.t. to geometry 1, using threshold 0.5
Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.7903153739890221
Attempting exact match with iRMSD threshold: 0.5
Could not find alignment below iRMSD threshold
Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.1880952492899788
Attempting exact match with iRMSD threshold: 0.1890952492899788
Could not find alignment below iRMSD threshold
Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.24909627819106586
Attempting exact match with iRMSD threshold: 0.25009627819106

In [17]:
ls -haltr sample_selected_TS-20250709-SCAN-9w-various_models/sample_selected_TS-20250709-SCAN-9w-various_models_aligned/

ls: sample_selected_TS-20250709-SCAN-9w-various_models/sample_selected_TS-20250709-SCAN-9w-various_models_aligned/: No such file or directory


In [18]:
output_file = "WL1-TS11193_aligned.xyz"
output_path = os.path.join(output_dir, output_file)
all_atoms = ase.io.read(output_path, index=':')
target = all_atoms[1]
base = all_atoms[-1]

FileNotFoundError: [Errno 2] No such file or directory: '/Users/trondlinjordet/Documents/MANABIYA_work/OAReactDiff/scratch/results/results/sample_selected_TS-20250709-SCAN-9w-various_models_aligned_20250724/WL1-TS11193_aligned.xyz'

In [ ]:
aligned_base, rmsd = align(base, target, threshold=0.5, verbose=True)
print(f'iRSMD={rmsd}')

In [ ]:
view_aligned(aligned_base, target)

# Multiple attempts at viewing multiple systems

In [ ]:
# Rewriting function to work with multiple views and Hbox

def view_aligned(aligned, target):
    v = nv.show_ase(aligned+target, viewer='ngl')
    v.remove_spacefill()
    # Try having the aligned (generated) geometry with sub 1.0 opacity around the reference geometry.  
    v.add_ball_and_stick(selection=range(len(aligned)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.2, radiusType='covalent',opacity=0.7)
    v.add_ball_and_stick(selection=range(len(aligned),len(aligned)+len(target)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.1, radiusType='covalent')
    return(v)

In [ ]:
views = []
for i, atoms_obj in enumerate(all_atoms[3:7]):
    # Create an NGLWidget for each Atoms object
    view = view_aligned(atoms_obj, all_atoms[1])
#    system_name = [os.path.basename(os.path.dirname(x.removesuffix('ddpm-epoch'))) for x in atoms_obj.info.keys() if x.endswith('ddpm-epoch')][0]
    system_name_parts = [key for key, val in atoms_obj.info.items() if val]
    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool):
            system_name_parts.append(f"{key} {val}")
    system_name = " ".join(system_name_parts)
    print(system_name)

    # Add a title to each view for clarity
    view.add_text(system_name, x=0, y=0, z=0, color='black', size=1.5)

    # Set initial camera perspective for the first view
    # This will be propagated to others by nglview.link()
    if i == 0:
        # You can programmatically set a camera orientation if desired
        # view.camera = 'orthographic' # or 'perspective'
        # view.center_view_only = True # Center view on the molecule, not the whole canvas
        pass # Let NGLView pick a default, or set manually in the first view after it loads

    views.append(view)
if len(views) > 1:
    nv.link(views)
    print("NGLView widgets linked for synchronized camera and perspective.")
else:
    print("Only one view created, no linking needed.")

# --- 5. Arrange and Display Widgets ---
# Use ipywidgets.HBox (horizontal box) or VBox (vertical box) to display them.

# Adjust layout for better display (optional)
for view in views:
    view.layout.width = '300px' # Set a fixed width for each view
    view.layout.height = '300px' # Set a fixed height for each view
    view.layout.border = '2px solid lightgray' # Add a border for visual separation

# Display horizontally
hbox_layout = widgets.HBox(views)
print("\nDisplaying visualizations:")
display(hbox_layout)

In [ ]:
# --- Corrected view_aligned function ---
def view_aligned(aligned, target):
    v = nv.show_ase(aligned + target, viewer='ngl') # Corrected: use nv.show_ase

    # CRITICAL FIX: All NGL operations on the viewer itself need 'v.view.'
    v.view.remove_all_components() # Start clean, remove default representations

    # Try having the aligned (generated) geometry with sub 1.0 opacity around the reference geometry.
    # Add representation for the 'aligned' part
    v.view.add_ball_and_stick(
        selection=f":{len(aligned)-1}", # Selects atoms from 0 to len(aligned)-1
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.2,
        radiusType='covalent',
        opacity=0.7
    )
    # Add representation for the 'target' (reference) part
    v.view.add_ball_and_stick(
        selection=f"{len(aligned)}:", # Selects atoms from len(aligned) to end
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.1,
        radiusType='covalent'
    )
    return v

# --- Main loop for creating views ---
views = []
# Using all_atoms[3:7] as per your code
# Ensure all_atoms[1] is the target reference for the aligned structures
target_reference_mol = all_atoms[1] # Store it once for clarity

for i, atoms_obj in enumerate(all_atoms[3:7]): # Loop through elements 3, 4, 5, 6
    # Create an NGLWidget for each Atoms object
    view_widget = view_aligned(atoms_obj, target_reference_mol) # Pass target_reference_mol

    # --- Logic for system_name extraction ---
    # This logic is complex and relies on very specific keys in atoms_obj.info
    # Let's refine it slightly to be safer
    system_name_parts = []
    # Find the part ending with 'ddpm-epoch' first
    ddpm_epoch_key = next((k for k in atoms_obj.info.keys() if k.endswith('ddpm-epoch')), None)
    if ddpm_epoch_key:
        path_val = atoms_obj.info[ddpm_epoch_key]
        # Your original logic: os.path.basename(os.path.dirname(path_val.removesuffix('ddpm-epoch')))
        # This implies path_val is a string like '/some/path/to/model/ddpm-epoch'
        try:
            # Safely get basename of dirname and remove suffix
            cleaned_path = path_val.removesuffix('ddpm-epoch')
            name_from_path = os.path.basename(os.path.dirname(cleaned_path))
            system_name_parts.append(name_from_path)
        except Exception as e:
            print(f"Warning: Error processing 'ddpm-epoch' key for system {i}: {e}")
            system_name_parts.append(f"ErrorPathName_{i}") # Fallback

    # Add other key-value pairs from info if value is not boolean and not already added
    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool) and key != ddpm_epoch_key: # Exclude boolean flags and the epoch key
            # Format depends on whether val is string or float etc.
            if isinstance(val, (float, int)):
                system_name_parts.append(f"{key}: {val:.3f}") # Format floats
            else:
                system_name_parts.append(f"{key}: {val}")

    # Fallback if no parts collected (e.g., if .info is empty or doesn't match criteria)
    if not system_name_parts:
        system_name = atoms_obj.info.get('name', f"Unnamed System {i+1}")
    else:
        system_name = " ".join(system_name_parts)

    print(system_name) # Print for debugging

    # CRITICAL FIX: Use view_widget.view.add_text
    view_widget.view.add_text(system_name, x=0, y=0, z=0, color='black', size=1.5)

    # Set initial camera perspective for the first view
    if i == 0:
        # These are usually applied to the 'view.view' object as well, or are widget properties.
        # For camera type: view_widget.camera = 'orthographic' (this is on the widget itself)
        pass # Keep this as is for camera properties, they usually apply to the widget directly.

    views.append(view_widget)

# --- Link views and Display ---
if len(views) > 1:
    nv.link(views)
    print("NGLView widgets linked for synchronized camera and perspective.")
else:
    print("Only one view created, no linking needed.")

for v in views: # Renamed 'view' to 'v' here to avoid conflict
    v.layout.width = '300px'
    v.layout.height = '300px'
    v.layout.border = '2px solid lightgray'

hbox_layout = widgets.HBox(views)
print("\nDisplaying visualizations:")
display(hbox_layout)

In [ ]:
# --- CORRECTED view_aligned function ---
def view_aligned(aligned, target):
    v = nv.show_ase(aligned + target, viewer='ngl')

    # CRITICAL FIX: Remove '.view' - methods are directly on 'v'
    v.remove_all_components() # Corrected call

    # Add representation for the 'aligned' part
    v.add_ball_and_stick( # Corrected call
        selection=f":{len(aligned)-1}",
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.2,
        radiusType='covalent',
        opacity=0.7
    )
    # Add representation for the 'target' (reference) part
    v.add_ball_and_stick( # Corrected call
        selection=f"{len(aligned)}:",
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.1,
        radiusType='covalent'
    )
    return v

# --- Main loop for creating views (corrected) ---
views = []
target_reference_mol = all_atoms[1]

for i, atoms_obj in enumerate(all_atoms[3:7]):
    view_widget = view_aligned(atoms_obj, target_reference_mol)

    # --- Logic for system_name extraction ---
    system_name_parts = []
    ddpm_epoch_key = next((k for k in atoms_obj.info.keys() if k.endswith('ddpm-epoch')), None)
    if ddpm_epoch_key:
        path_val = atoms_obj.info[ddpm_epoch_key]
        try:
            cleaned_path = path_val.removesuffix('ddpm-epoch')
            name_from_path = os.path.basename(os.path.dirname(cleaned_path))
            system_name_parts.append(name_from_path)
        except Exception as e:
            print(f"Warning: Error processing 'ddpm-epoch' key for system {i}: {e}")
            system_name_parts.append(f"ErrorPathName_{i}")

    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool) and key != ddpm_epoch_key:
            if isinstance(val, (float, int)):
                system_name_parts.append(f"{key}: {val:.3f}")
            else:
                system_name_parts.append(f"{key}: {val}")

    if not system_name_parts:
        system_name = atoms_obj.info.get('name', f"Unnamed System {i+1}")
    else:
        system_name = " ".join(system_name_parts)

    print(system_name)

    # CRITICAL FIX: Remove '.view' - method is directly on 'view_widget'
    view_widget.add_text(system_name, x=0, y=0, z=0, color='black', size=1.5) # Corrected call

    # Camera perspective settings are usually applied directly to the NGLWidget object
    # For example: view_widget.camera = 'orthographic'
    # No change needed here for this specific part from your original code.
    if i == 0:
        pass

    views.append(view_widget)

# --- Link views and Display ---
if len(views) > 1:
    nv.link(views)
    print("NGLView widgets linked for synchronized camera and perspective.")
else:
    print("Only one view created, no linking needed.")

for v_item in views: # Changed loop variable to v_item to avoid confusion
    v_item.layout.width = '300px'
    v_item.layout.height = '300px'
    v_item.layout.border = '2px solid lightgray'

hbox_layout = widgets.HBox(views)
print("\nDisplaying visualizations:")
display(hbox_layout)

In [ ]:
# --- CORRECTED view_aligned function ---
def view_aligned(aligned, target):
    v = nv.show_ase(aligned + target, viewer='ngl')

    # CRITICAL FIX: Use clear_representations() to remove existing representations
    v.clear_representations() # Corrected method call

    # Add representation for the 'aligned' part
    v.add_ball_and_stick(
        selection=f":{len(aligned)-1}",
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.2,
        radiusType='covalent',
        opacity=0.7
    )
    # Add representation for the 'target' (reference) part
    v.add_ball_and_stick(
        selection=f"{len(aligned)}:",
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.1,
        radiusType='covalent'
    )
    return v

# --- Main loop for creating views (corrected) ---
views = []
target_reference_mol = all_atoms[1]

for i, atoms_obj in enumerate(all_atoms[3:7]):
    view_widget = view_aligned(atoms_obj, target_reference_mol)

    # --- Logic for system_name extraction ---
    system_name_parts = []
    ddpm_epoch_key = next((k for k in atoms_obj.info.keys() if k.endswith('ddpm-epoch')), None)
    if ddpm_epoch_key:
        path_val = atoms_obj.info[ddpm_epoch_key]
        try:
            cleaned_path = path_val.removesuffix('ddpm-epoch')
            name_from_path = os.path.basename(os.path.dirname(cleaned_path))
            system_name_parts.append(name_from_path)
        except Exception as e:
            print(f"Warning: Error processing 'ddpm-epoch' key for system {i}: {e}")
            system_name_parts.append(f"ErrorPathName_{i}")

    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool) and key != ddpm_epoch_key:
            if isinstance(val, (float, int)):
                system_name_parts.append(f"{key}: {val:.3f}")
            else:
                system_name_parts.append(f"{key}: {val}")

    if not system_name_parts:
        system_name = atoms_obj.info.get('name', f"Unnamed System {i+1}")
    else:
        system_name = " ".join(system_name_parts)

    print(system_name)

    # Add text to the widget (directly, not via .view)
    view_widget.add_text(system_name, x=0, y=0, z=0, color='black', size=1.5)

    # Camera perspective settings are usually applied directly to the NGLWidget object
    if i == 0:
        pass

    views.append(view_widget)

# --- Link views and Display ---
if len(views) > 1:
    nv.link(views)
    print("NGLView widgets linked for synchronized camera and perspective.")
else:
    print("Only one view created, no linking needed.")

for v_item in views:
    v_item.layout.width = '300px'
    v_item.layout.height = '300px'
    v_item.layout.border = '2px solid lightgray'

hbox_layout = widgets.HBox(views)
print("\nDisplaying visualizations:")
display(hbox_layout)

In [ ]:
# --- CORRECTED view_aligned function (based on NGLView 3.x direct API) ---
def view_aligned(aligned, target):
    v = nv.show_ase(aligned + target, viewer='ngl')

    # Use clear_representations() to remove existing representations
    # This assumes NGLWidget has this method directly.
    # If this line causes an AttributeError, it confirms a core issue with the NGLWidget instance.
    v.clear_representations()

    # Add representation for the 'aligned' part
    v.add_ball_and_stick(
        selection=f":{len(aligned)-1}",
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.2,
        radiusType='covalent',
        opacity=0.7
    )
    # Add representation for the 'target' (reference) part
    v.add_ball_and_stick(
        selection=f"{len(aligned)}:",
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.1,
        radiusType='covalent'
    )
    return v

# --- Main loop for creating views ---
views = []
target_reference_mol = all_atoms[1]

for i, atoms_obj in enumerate(all_atoms[3:7]):
    view_widget = view_aligned(atoms_obj, target_reference_mol)

    # --- VERIFIED system_name extraction logic ---
    system_name_parts = []
    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool):
            system_name_parts.append(f"{key} {val}")
        elif val:
            system_name_parts.append(key)
    system_name = " ".join(system_name_parts)
    print(system_name)

    # --- DIAGNOSTIC CODE (Highly Recommended to Run This First) ---
    # If the `AttributeError: 'NGLWidget' object has no attribute 'add_text'`
    # persists, these lines will tell us what the object actually is.
    print(f"\n--- Diagnosing NGLWidget for system {i} ---")
    print(f"Type of view_widget: {type(view_widget)}")
    print(f"Is it an nglview.NGLWidget? {isinstance(view_widget, nv.NGLWidget)}")
    print(f"Does it have 'add_text' attribute? {'add_text' in dir(view_widget)}")
    print(f"Does it have 'clear_representations' attribute? {'clear_representations' in dir(view_widget)}")
    print(f"--- End Diagnosis ---\n")
    # --- END DIAGNOSTIC CODE ---

    # Add text to the widget (This is the line causing the current AttributeError)
    # This line depends on 'add_text' being an attribute of view_widget.
    # If the diagnostic above shows 'add_text' is False, this line will fail.
    #view_widget.add_text(system_name, x=0, y=0, z=0, color='black', size=1.5)
    view_widget.add_representation(repr_type="label", name = "label", showBackground = True, labelType = "atomname", color = "black", xOffset = 0.5 , zOffset =5 )


    if i == 0:
        pass # Camera perspective settings etc.

    views.append(view_widget)

# --- Link views and Display ---
if len(views) > 1:
    nv.link(views)
    print("NGLView widgets linked for synchronized camera and perspective.")
else:
    print("Only one view created, no linking needed.")

for v_item in views:
    v_item.layout.width = '300px'
    v_item.layout.height = '300px'
    v_item.layout.border = '2px solid lightgray'

hbox_layout = widgets.HBox(views)
print("\nDisplaying visualizations:")
display(hbox_layout)

In [ ]:
# --- view_aligned function with ONLY necessary changes, selection arguments reverted ---
def view_aligned(aligned, target):
    v = nv.show_ase(aligned + target, viewer='ngl')

    # This was changed from .remove_all_components() to clear_representations()
    # because .remove_all_components() caused an AttributeError, and
    # your diagnostic confirmed clear_representations() is available.
    v.clear_representations()

    # selection arguments are reverted to your original 'range()' syntax.
    v.add_ball_and_stick(
        selection=range(len(aligned)), # REVERTED: Your original syntax
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.2,
        radiusType='covalent',
        opacity=0.7
    )
    # selection arguments are reverted to your original 'range()' syntax.
    v.add_ball_and_stick(
        selection=range(len(aligned),len(aligned)+len(target)), # REVERTED: Your original syntax
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.1,
        radiusType='covalent'
    )
    return v

# --- Main loop for creating views and titles (rest of the changes are for confirmed errors) ---
individual_display_boxes = []
views_for_linking = []
target_reference_mol = all_atoms[1]

for i, atoms_obj in enumerate(all_atoms[3:7]):
    view_widget = view_aligned(atoms_obj, target_reference_mol)

    # --- VERIFIED system_name extraction logic ---
    # This section is YOUR VERIFIED code for system name extraction.
    system_name_parts = []
    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool):
            system_name_parts.append(f"{key} {val}")
        elif val:
            system_name_parts.append(key)
    system_name = " ".join(system_name_parts)
    print(system_name)

    # Diagnostic output removed to clean up, but its findings (add_text=False) are crucial.

    # --- Handling add_text (removed due to AttributeError) ---
    # As 'add_text' is confirmed to be missing from NGLWidget,
    # we use an ipywidgets.Label for the title.
    name_label = widgets.Label(value=system_name)
    name_label.layout.padding = '5px'

    # Set layout for the NGLView widget
    view_widget.layout.width = '300px'
    view_widget.layout.height = '300px'
    view_widget.layout.border = '2px solid lightgray'

    # Combine label and NGLView widget in a vertical box
    combined_box = widgets.VBox([name_label, view_widget])
    combined_box.layout.margin = '5px'

    individual_display_boxes.append(combined_box)
    views_for_linking.append(view_widget)

    if i == 0:
        pass

# --- CORRECTED: Link NGLView widgets using ipywidgets.jslink ---
# This replaced nv.link() because nv.link() caused an AttributeError.
if len(views_for_linking) > 1:
    master_view = views_for_linking[0]
    for i in range(1, len(views_for_linking)):
        slave_view = views_for_linking[i]
        widgets.jslink((master_view, 'camera'), (slave_view, 'camera'))
    print("NGLView widgets linked for synchronized camera and perspective using jslink.")
else:
    print("Only one view created, no linking needed.")

# --- Arrange and Display Widgets ---
hbox_layout = widgets.HBox(individual_display_boxes)
hbox_layout.layout.flex_flow = 'row wrap'
hbox_layout.layout.align_items = 'flex-start'

print("\nDisplaying visualizations:")
display(hbox_layout)

In [ ]:
# --- view_aligned function with ONLY necessary changes, selection arguments reverted ---
def view_aligned(aligned, target):
    v = nv.show_ase(aligned + target, viewer='ngl')

    # This was changed from .remove_all_components() to clear_representations()
    # because .remove_all_components() caused an AttributeError, and
    # your diagnostic confirmed clear_representations() is available.
    v.clear_representations()

    # selection arguments are reverted to your original 'range()' syntax.
    v.add_ball_and_stick(
        selection=range(len(aligned)), # REVERTED: Your original syntax
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.2,
        radiusType='covalent',
        opacity=0.7
    )
    # selection arguments are reverted to your original 'range()' syntax.
    v.add_ball_and_stick(
        selection=range(len(aligned),len(aligned)+len(target)), # REVERTED: Your original syntax
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.1,
        radiusType='covalent'
    )
    return v

# --- Main loop for creating views and titles ---
individual_display_boxes = []
views_for_linking = []
target_reference_mol = all_atoms[1]

for i, atoms_obj in enumerate(all_atoms[3:7]):
    view_widget = view_aligned(atoms_obj, target_reference_mol)

    # --- VERIFIED system_name extraction logic ---
    system_name_parts = []
    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool):
            system_name_parts.append(f"{key} {val}")
        elif val:
            system_name_parts.append(key)
    system_name = " ".join(system_name_parts)
    print(system_name)

    # --- Handling add_text (removed due to AttributeError) ---
    # As 'add_text' is confirmed to be missing from NGLWidget,
    # we use an ipywidgets.Label for the title.
    name_label = widgets.Label(value=system_name)
    name_label.layout.padding = '5px'

    # Set layout for the NGLView widget
    view_widget.layout.width = '300px'
    view_widget.layout.height = '300px'
    view_widget.layout.border = '2px solid lightgray'

    # Combine label and NGLView widget in a vertical box
    combined_box = widgets.VBox([name_label, view_widget])
    combined_box.layout.margin = '5px'

    individual_display_boxes.append(combined_box)
    views_for_linking.append(view_widget)

    if i == 0:
        pass

# --- CORRECTED: Link NGLView widgets using ipywidgets.jslink with '_camera_orientation' ---
# This replaced nv.link() because nv.link() caused an AttributeError.
# This replaces 'camera' with '_camera_orientation' because 'camera' caused a TypeError.
if len(views_for_linking) > 1:
    master_view = views_for_linking[0]
    for i in range(1, len(views_for_linking)):
        slave_view = views_for_linking[i]
        # CHANGE HERE: Use '_camera_orientation' instead of 'camera'
        widgets.jslink((master_view, '_camera_orientation'), (slave_view, '_camera_orientation'))
    print("NGLView widgets linked for synchronized camera and perspective using jslink.")
else:
    print("Only one view created, no linking needed.")

# --- Arrange and Display Widgets ---
hbox_layout = widgets.HBox(individual_display_boxes)
hbox_layout.layout.flex_flow = 'row wrap'
hbox_layout.layout.align_items = 'flex-start'

print("\nDisplaying visualizations:")
display(hbox_layout)

In [ ]:
# --- view_aligned function with specific visibility and centering changes ---
def view_aligned(aligned, target):
    v = nv.show_ase(aligned + target, viewer='ngl')

    # Specific Change 1: Ensure camera centers on molecule
    v.center_view_only = True
    # Specific Change 3: Set a background color for better contrast
    v.background = 'white' # Or 'black' if you prefer

    v.clear_representations()

    v.add_ball_and_stick(
        selection=range(len(aligned)),
        cylinderOnly=False,
        aspectRatio=3.0,
        # Specific Change 2a: Increased radiusScale for better visibility
        radiusScale=0.5,
        radiusType='covalent',
        opacity=0.7
    )
    v.add_ball_and_stick(
        selection=range(len(aligned),len(aligned)+len(target)),
        cylinderOnly=False,
        aspectRatio=3.0,
        # Specific Change 2b: Increased radiusScale for better visibility
        radiusScale=0.3,
        radiusType='covalent'
    )
    return v

# --- Main loop for creating views and titles ---
individual_display_boxes = []
views_for_linking = []
target_reference_mol = all_atoms[1]

for i, atoms_obj in enumerate(all_atoms[3:7]):
    view_widget = view_aligned(atoms_obj, target_reference_mol)

    # --- VERIFIED system_name extraction logic ---
    system_name_parts = []
    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool):
            system_name_parts.append(f"{key} {val}")
        elif val:
            system_name_parts.append(key)
    system_name = " ".join(system_name_parts)
    print(system_name)

    name_label = widgets.Label(value=system_name)
    name_label.layout.padding = '5px'

    view_widget.layout.width = '300px'
    view_widget.layout.height = '300px'
    view_widget.layout.border = '2px solid lightgray'

    combined_box = widgets.VBox([name_label, view_widget])
    combined_box.layout.margin = '5px'

    individual_display_boxes.append(combined_box)
    views_for_linking.append(view_widget)

    if i == 0:
        pass

# --- Link NGLView widgets using ipywidgets.jslink with '_camera_orientation' ---
if len(views_for_linking) > 1:
    master_view = views_for_linking[0]
    for i in range(1, len(views_for_linking)):
        slave_view = views_for_linking[i]
        widgets.jslink((master_view, '_camera_orientation'), (slave_view, '_camera_orientation'))
    print("NGLView widgets linked for synchronized camera and perspective using jslink.")
else:
    print("Only one view created, no linking needed.")

# --- Arrange and Display Widgets ---
hbox_layout = widgets.HBox(individual_display_boxes)
hbox_layout.layout.flex_flow = 'row wrap'
hbox_layout.layout.align_items = 'flex-start'

print("\nDisplaying visualizations:")
display(hbox_layout)








In [ ]:
view_widget = view_aligned(atoms_obj, target_reference_mol)
view_widget

In [ ]:
# --- view_aligned function with specific visibility and centering changes, and reverted radiusScale ---
def view_aligned(aligned, target):
    v = nv.show_ase(aligned + target, viewer='ngl')

    # Ensure camera centers on molecule
    v.center_view_only = True
    # Set a background color for better contrast
    v.background = 'white' # Or 'black' if you prefer

    v.clear_representations()

    v.add_ball_and_stick(
        selection=range(len(aligned)),
        cylinderOnly=False,
        aspectRatio=3.0,
        # REVERTED: Original radiusScale for aligned part
        radiusScale=0.2,
        radiusType='covalent',
        opacity=0.7
    )
    v.add_ball_and_stick(
        selection=range(len(aligned),len(aligned)+len(target)),
        cylinderOnly=False,
        aspectRatio=3.0,
        # REVERTED: Original radiusScale for target part
        radiusScale=0.1,
        radiusType='covalent'
    )
    return v

# --- Main loop for creating views and titles ---
individual_display_boxes = []
views_for_linking = []
# Assuming target_reference_mol and all_atoms are defined elsewhere as per your request
# target_reference_mol = all_atoms[1]

for i, atoms_obj in enumerate(all_atoms[3:7]):
    view_widget = view_aligned(atoms_obj, target_reference_mol)

    # --- VERIFIED system_name extraction logic ---
    system_name_parts = []
    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool):
            system_name_parts.append(f"{key} {val}")
        elif val:
            system_name_parts.append(key)
    system_name = " ".join(system_name_parts)
    print(system_name)

    name_label = widgets.Label(value=system_name)
    name_label.layout.padding = '5px'

    view_widget.layout.width = '300px'
    view_widget.layout.height = '300px'
    view_widget.layout.border = '2px solid lightgray'

    combined_box = widgets.VBox([name_label, view_widget])
    combined_box.layout.margin = '5px'

    individual_display_boxes.append(combined_box)
    views_for_linking.append(view_widget)

    if i == 0:
        pass

# --- Link NGLView widgets using ipywidgets.jslink with '_camera_orientation' ---
if len(views_for_linking) > 1:
    master_view = views_for_linking[0]
    for i in range(1, len(views_for_linking)):
        slave_view = views_for_linking[i]
        widgets.jslink((master_view, '_camera_orientation'), (slave_view, '_camera_orientation'))
    print("NGLView widgets linked for synchronized camera and perspective using jslink.")
else:
    print("Only one view created, no linking needed.")

# --- Arrange and Display Widgets ---
hbox_layout = widgets.HBox(individual_display_boxes)
hbox_layout.layout.flex_flow = 'row wrap'
hbox_layout.layout.align_items = 'flex-start'

print("\nDisplaying visualizations:")
display(hbox_layout)

# Current attempt

In [ ]:
import os
import nglview as nv
import ipywidgets as widgets
from IPython.display import display
from ase import Atoms
from ase.build import molecule


# --- view_aligned function with specific visibility and centering changes, and reverted radiusScale ---
def view_aligned(aligned, target):
    v = nv.show_ase(aligned + target, viewer='ngl')

    # Ensure camera centers on molecule
    v.center_view_only = True
    # Set a background color for better contrast
    v.background = 'white' # Or 'black' if you prefer

    v.clear_representations()

    v.add_ball_and_stick(
        selection=range(len(aligned)),
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.2, # Reverted to your original value
        radiusType='covalent',
        opacity=0.7
    )
    v.add_ball_and_stick(
        selection=range(len(aligned),len(aligned)+len(target)),
        cylinderOnly=False,
        aspectRatio=3.0,
        radiusScale=0.1, # Reverted to your original value
        radiusType='covalent'
    )
    return v

# --- Main loop for creating views and titles ---
individual_display_boxes = []
views_for_linking = []
# You need to ensure 'all_atoms' and 'target_reference' are defined in your script before this loop
# Example (from previous iterations):
target_reference = all_atoms[1]

for i, atoms_obj in enumerate(all_atoms[3:7]): # Assuming all_atoms is defined and populated
    view_widget = view_aligned(atoms_obj, target_reference)

    # --- VERIFIED system_name extraction logic ---
    system_name_parts = []
    for key, val in atoms_obj.info.items():
        if not isinstance(val, bool):
            system_name_parts.append(f"{key} {val}")
        elif val:
            system_name_parts.append(key)
    system_name = " ".join(system_name_parts)
    print(system_name)

    name_label = widgets.Label(value=system_name)
    name_label.layout.padding = '5px'

    view_widget.layout.width = '300px'
    view_widget.layout.height = '300px'
    view_widget.layout.border = '2px solid lightgray'

    combined_box = widgets.VBox([name_label, view_widget])
    combined_box.layout.margin = '5px'

    individual_display_boxes.append(combined_box)
    views_for_linking.append(view_widget)

    if i == 0:
        pass

# --- Link NGLView widgets using ipywidgets.jslink with '_camera_orientation' ---
if len(views_for_linking) > 1:
    master_view = views_for_linking[0]
    for i in range(1, len(views_for_linking)):
        slave_view = views_for_linking[i]
        widgets.jslink((master_view, '_camera_orientation'), (slave_view, '_camera_orientation'))
    print("NGLView widgets linked for synchronized camera and perspective using jslink.")
else:
    print("Only one view created, no linking needed.")

# --- Arrange and Display Widgets ---
hbox_layout = widgets.HBox(individual_display_boxes)
hbox_layout.layout.flex_flow = 'row wrap'
hbox_layout.layout.align_items = 'flex-start'

print("\nDisplaying visualizations:")
display(hbox_layout)

In [ ]:
views_for_linking[0]

In [ ]:
view_widget = view_aligned(atoms_obj, target_reference)
view_widget

In [ ]:
atoms_obj.info.keys()

In [ ]:
atoms_obj.info

In [ ]:
x = '/misc/home/guest50/OAReactDiff/oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/9w-lr5e-4-ValFix4-SCAN-leftnet64449ad28e23/ddpm-epoch'
os.path.basename(os.path.dirname(x.removesuffix('ddpm-epoch')))

In [ ]:
foolist = [os.path.basename(os.path.dirname(x.removesuffix('ddpm-epoch'))) for x in atoms_obj.info.keys() if x.endswith('ddpm-epoch')]
foolist

In [ ]:
output_dir